# SALESPULSE Sales Management Portal
A simple console portal to track deals through **Pipeline -> Qualified -> Closed (Won / Lost)**.

**Login:** username `admin`, password `sales123`

**Menu:** Sales Dashboard, Deals in Pipeline, Qualified Deals, Closed Deals, Manage Deals (add / search / move stage / update / delete), Sales Report

**Files created:** `deals.json` (saved deals), `activity_log.txt` (appended on every action), `sales_report.txt` (report)

The output under the code cell is a sample run. Run the cell yourself and type your own answers.

In [1]:
# SALESPULSE Sales Management Portal (simplified version)

import json

FILE_NAME = "deals.json"
LOG_NAME = "activity_log.txt"
REPORT_NAME = "sales_report.txt"

stages = ["Pipeline", "Qualified", "Closed Won", "Closed Lost"]
chances = {"Pipeline": 0.2, "Qualified": 0.5}
products = ["CRM Software", "Cloud Hosting", "Analytics Suite",
            "Training Package", "Support Contract"]
reps = ["Sara", "Bilal", "Hamza"]
TARGET = 3000000

sample_deals = [
    {"id": 1, "company": "Nexa Textiles", "contact": "Ayesha Khan", "product": "Analytics Suite", "value": 450000, "stage": "Pipeline", "rep": "Sara", "close_date": "2026-12-20", "note": "Intro call done"},
    {"id": 2, "company": "Punjab Logistics", "contact": "Imran Sheikh", "product": "Cloud Hosting", "value": 320000, "stage": "Pipeline", "rep": "Bilal", "close_date": "2026-12-15", "note": ""},
    {"id": 3, "company": "BrightPath Academy", "contact": "Maryam Tariq", "product": "Training Package", "value": 180000, "stage": "Pipeline", "rep": "Hamza", "close_date": "2027-01-10", "note": ""},
    {"id": 4, "company": "Zenith Pharma", "contact": "Omar Farooq", "product": "CRM Software", "value": 750000, "stage": "Pipeline", "rep": "Sara", "close_date": "2026-12-05", "note": "Wants a demo"},
    {"id": 5, "company": "CloudNine Solutions", "contact": "Usman Raza", "product": "Support Contract", "value": 900000, "stage": "Qualified", "rep": "Hamza", "close_date": "2026-11-28", "note": "Budget approved"},
    {"id": 6, "company": "Metro Healthcare", "contact": "Sadia Noor", "product": "CRM Software", "value": 680000, "stage": "Qualified", "rep": "Sara", "close_date": "2026-12-10", "note": ""},
    {"id": 7, "company": "Sunrise Foods", "contact": "Kamran Ali", "product": "Analytics Suite", "value": 380000, "stage": "Qualified", "rep": "Bilal", "close_date": "2026-11-25", "note": "Contract with legal"},
    {"id": 8, "company": "Karakoram Tech", "contact": "Zara Hussain", "product": "Cloud Hosting", "value": 520000, "stage": "Qualified", "rep": "Hamza", "close_date": "2026-12-18", "note": ""},
    {"id": 9, "company": "AlHamra Builders", "contact": "Tahir Mehmood", "product": "Support Contract", "value": 280000, "stage": "Closed Won", "rep": "Sara", "close_date": "2026-11-04", "note": ""},
    {"id": 10, "company": "Orbit Telecom", "contact": "Nadia Siddiqui", "product": "CRM Software", "value": 820000, "stage": "Closed Won", "rep": "Bilal", "close_date": "2026-10-20", "note": ""},
    {"id": 11, "company": "GreenLeaf Agro", "contact": "Faisal Javed", "product": "Training Package", "value": 150000, "stage": "Closed Won", "rep": "Hamza", "close_date": "2026-10-27", "note": ""},
    {"id": 12, "company": "Falcon Security", "contact": "Rabia Anwar", "product": "Cloud Hosting", "value": 410000, "stage": "Closed Lost", "rep": "Sara", "close_date": "2026-10-30", "note": "Chose a cheaper competitor"},
    {"id": 13, "company": "Crescent Events", "contact": "Asad Mirza", "product": "Analytics Suite", "value": 260000, "stage": "Closed Lost", "rep": "Bilal", "close_date": "2026-10-25", "note": "Budget frozen"},
]


# ---------- helper functions ----------

def get_number(message):
    while True:
        try:
            number = float(input(message))
            if number <= 0:
                print("Enter a number greater than 0.")
            else:
                if number == int(number):
                    number = int(number)
                return number
        except ValueError:
            print("Invalid input. Enter a number.")


def get_integer(message):
    while True:
        try:
            number = int(input(message))
            if number < 0:
                print("Enter a number that is 0 or more.")
            else:
                return number
        except ValueError:
            print("Invalid input. Enter a whole number.")


def get_text(message):
    text = input(message).strip()
    while text == "":
        print("This cannot be empty.")
        text = input(message).strip()
    return text


def get_date(message):
    while True:
        text = input(message).strip()
        parts = text.split("-")
        try:
            if len(parts) == 3 and len(parts[0]) == 4:
                int(parts[0])
                month = int(parts[1])
                day = int(parts[2])
                if month >= 1 and month <= 12 and day >= 1 and day <= 31:
                    return text
        except ValueError:
            pass
        print("Enter the date like this: 2026-12-31")


def choose(title, options):
    print("\n" + title)
    for i in range(len(options)):
        print(i + 1, ".", options[i])
    choice = get_integer("Choose: ")
    while choice < 1 or choice > len(options):
        print("Choose a number from the list.")
        choice = get_integer("Choose: ")
    return options[choice - 1]


def next_id(records):
    biggest = 0
    for record in records:
        if record["id"] > biggest:
            biggest = record["id"]
    return biggest + 1


def find_by_id(records, record_id):
    for record in records:
        if record["id"] == record_id:
            return record
    return None


def show_deal_line(d):
    print(d["id"], "|", d["company"], "|", d["product"], "| Rs.", d["value"],
          "|", d["stage"], "|", d["rep"], "|", d["close_date"])


# ---------- file handling ----------

def save_deals(data):
    try:
        with open(FILE_NAME, "w") as file:
            json.dump(data, file, indent=2)
        return True
    except PermissionError:
        print("No permission to save the file. Changes were not saved.")
        return False


def load_deals():
    try:
        with open(FILE_NAME, "r") as file:
            return json.load(file)
    except FileNotFoundError:
        print("No saved file found. Creating one with sample deals.")
        save_deals(sample_deals)
        return sample_deals
    except PermissionError:
        print("No permission to read the file. Using sample deals.")
        return sample_deals
    except ValueError:
        print("Saved file is damaged. Starting with sample deals.")
        return sample_deals


def write_log(text):
    try:
        with open(LOG_NAME, "a") as file:
            file.write(text + "\n")
    except PermissionError:
        print("Cannot write to the log file.")


deals = load_deals()


# ---------- login ----------

def user_login():
    print("\n===== SALESPULSE LOGIN =====")
    print("Username: admin | Password: sales123")
    attempts = 3
    while attempts > 0:
        username = input("Username: ")
        password = input("Password: ")
        if username == "admin" and password == "sales123":
            print("Login successful!")
            write_log("Admin logged in")
            return True
        attempts -= 1
        print("Wrong login. Attempts left:", attempts)
    print("Access denied.")
    return False


# ---------- dashboard ----------

def show_dashboard():
    open_count = 0
    open_value = 0
    forecast = 0
    won_count = 0
    won_value = 0
    lost_count = 0
    for d in deals:
        if d["stage"] == "Closed Won":
            won_count += 1
            won_value += d["value"]
        elif d["stage"] == "Closed Lost":
            lost_count += 1
        else:
            open_count += 1
            open_value += d["value"]
            forecast += d["value"] * chances[d["stage"]]

    print("\n===== SALESPULSE DASHBOARD =====")
    print("Total deals      :", len(deals))
    print("Open deals       :", open_count)
    print("Pipeline value   : Rs.", open_value)
    print("Forecast         : Rs.", round(forecast))
    print("Won revenue      : Rs.", won_value, "(", won_count, "deals )")
    print("Lost deals       :", lost_count)
    if won_count + lost_count > 0:
        print("Win rate         :", round(won_count / (won_count + lost_count) * 100, 1), "%")

    percent = won_value / TARGET * 100
    blocks = int(percent / 5)
    if blocks > 20:
        blocks = 20
    print("\n--- Target: Rs.", TARGET, "---")
    print("[" + "#" * blocks + "-" * (20 - blocks) + "]", round(percent, 1), "%")

    print("\n--- Deals by stage ---")
    for stage in stages:
        count = 0
        total = 0
        for d in deals:
            if d["stage"] == stage:
                count += 1
                total += d["value"]
        print(stage.ljust(12), count, "deals | Rs.", total, "|", "#" * int(total / 100000))

    print("\n--- Sales team ---")
    for rep in reps:
        won = 0
        still_open = 0
        for d in deals:
            if d["rep"] == rep:
                if d["stage"] == "Closed Won":
                    won += d["value"]
                elif d["stage"] != "Closed Lost":
                    still_open += d["value"]
        print(rep.ljust(8), "won Rs.", won, "| open Rs.", still_open)

    biggest = None
    for d in deals:
        if d["stage"] == "Pipeline" or d["stage"] == "Qualified":
            if biggest is None or d["value"] > biggest["value"]:
                biggest = d
    if biggest is not None:
        print("\nBiggest open deal:", biggest["company"], "- Rs.", biggest["value"])


# ---------- deal views ----------

def show_stage(stage_list, title):
    print("\n===== " + title + " =====")
    count = 0
    total = 0
    for d in deals:
        if d["stage"] in stage_list:
            show_deal_line(d)
            if d["stage"] == "Closed Lost":
                print("     Reason lost:", d["note"])
            count += 1
            total += d["value"]
    if count == 0:
        print("No deals here.")
    print("Deals:", count, "| Total value: Rs.", total)


def show_all_deals():
    print("\n===== ALL DEALS =====")
    for d in deals:
        show_deal_line(d)


# ---------- deals (add, search, update, delete) ----------

def add_record():
    company = get_text("Company name: ")
    contact = get_text("Contact person: ")
    product = choose("Select product:", products)
    value = get_number("Deal value (Rs.): ")
    stage = choose("Starting stage:", ["Pipeline", "Qualified"])
    rep = choose("Sales rep:", reps)
    close_date = get_date("Expected close date (YYYY-MM-DD): ")
    note = input("Note (optional): ")
    deal = {"id": next_id(deals), "company": company, "contact": contact,
            "product": product, "value": value, "stage": stage, "rep": rep,
            "close_date": close_date, "note": note}
    deals.append(deal)
    save_deals(deals)
    write_log("Added deal " + str(deal["id"]) + " " + company)
    print("Deal added!")
    return deal


def search_record():
    word = get_text("Search company, contact, product or rep: ").lower()
    found = []
    for d in deals:
        if word in d["company"].lower() or word in d["contact"].lower() \
                or word in d["product"].lower() or word in d["rep"].lower():
            found.append(d)
    if len(found) == 0:
        print("No deal found.")
    for d in found:
        show_deal_line(d)
    return found


def move_stage():
    show_all_deals()
    d = find_by_id(deals, get_integer("Deal ID to move: "))
    if d is None:
        print("Deal not found.")
        return
    new_stage = choose("Move to stage:", stages)
    if new_stage == "Closed Won":
        d["value"] = get_number("Final deal value (Rs.): ")
    elif new_stage == "Closed Lost":
        d["note"] = get_text("Reason for losing the deal: ")
    d["stage"] = new_stage
    save_deals(deals)
    write_log("Deal " + str(d["id"]) + " moved to " + new_stage)
    print("Deal moved to", new_stage)


def update_deal():
    show_all_deals()
    d = find_by_id(deals, get_integer("Deal ID to update: "))
    if d is None:
        print("Deal not found.")
        return
    d["value"] = get_number("New deal value (Rs.): ")
    d["close_date"] = get_date("New close date (YYYY-MM-DD): ")
    save_deals(deals)
    write_log("Updated deal " + str(d["id"]))
    print("Deal updated!")


def delete_deal():
    show_all_deals()
    d = find_by_id(deals, get_integer("Deal ID to delete: "))
    if d is None:
        print("Deal not found.")
        return
    deals.remove(d)
    save_deals(deals)
    write_log("Deleted deal " + str(d["id"]))
    print("Deal deleted!")


# ---------- report ----------

def generate_report():
    lines = ["===== SALESPULSE SALES REPORT ====="]
    won_value = 0
    for d in deals:
        if d["stage"] == "Closed Won":
            won_value += d["value"]
    lines.append("Total deals : " + str(len(deals)))
    lines.append("Won revenue : Rs. " + str(won_value))
    lines.append("Target      : " + str(round(won_value / TARGET * 100, 1)) + " % achieved")
    for stage in stages:
        lines.append("")
        lines.append("--- " + stage + " ---")
        total = 0
        for d in deals:
            if d["stage"] == stage:
                lines.append(d["company"] + " | " + d["rep"] + " | Rs. " + str(d["value"]))
                total += d["value"]
        lines.append("Total: Rs. " + str(total))
    report = "\n".join(lines)
    try:
        with open(REPORT_NAME, "w") as file:
            file.write(report)
        print("Report saved to", REPORT_NAME)
    except PermissionError:
        print("Could not save the report file.")
    return report


# ---------- menus ----------

def deals_menu():
    while True:
        print("\n--- MANAGE DEALS ---")
        print("1. View all  2. Add  3. Search  4. Move stage")
        print("5. Update value/date  6. Delete  7. Back")
        choice = input("Choice: ")
        if choice == "1":
            show_all_deals()
        elif choice == "2":
            add_record()
        elif choice == "3":
            search_record()
        elif choice == "4":
            move_stage()
        elif choice == "5":
            update_deal()
        elif choice == "6":
            delete_deal()
        elif choice == "7":
            break
        else:
            print("Invalid choice.")


def main():
    print("=" * 40)
    print("  WELCOME TO SALESPULSE SALES PORTAL")
    print("       Track. Qualify. Close.")
    print("=" * 40)

    if not user_login():
        return

    try:
        while True:
            print("\n===== MAIN MENU =====")
            print("1. Sales Dashboard")
            print("2. Deals in Pipeline")
            print("3. Qualified Deals")
            print("4. Closed Deals")
            print("5. Manage Deals")
            print("6. Sales Report")
            print("7. Exit")
            choice = input("Choice: ")
            if choice == "1":
                show_dashboard()
            elif choice == "2":
                show_stage(["Pipeline"], "DEALS IN PIPELINE")
            elif choice == "3":
                show_stage(["Qualified"], "QUALIFIED DEALS")
            elif choice == "4":
                show_stage(["Closed Won", "Closed Lost"], "CLOSED DEALS")
            elif choice == "5":
                deals_menu()
            elif choice == "6":
                print(generate_report())
            elif choice == "7":
                print("Thank you for using SALESPULSE!")
                break
            else:
                print("Invalid choice. Select 1 to 7.")
    finally:
        save_deals(deals)
        write_log("Admin logged out")


main()

No saved file found. Creating one with sample deals.
  WELCOME TO SALESPULSE SALES PORTAL
       Track. Qualify. Close.

===== SALESPULSE LOGIN =====
Username: admin | Password: sales123
Username: admin
Password: wrong
Wrong login. Attempts left: 2
Username: admin
Password: sales123
Login successful!

===== MAIN MENU =====
1. Sales Dashboard
2. Deals in Pipeline
3. Qualified Deals
4. Closed Deals
5. Manage Deals
6. Sales Report
7. Exit
Choice: 1

===== SALESPULSE DASHBOARD =====
Total deals      : 13
Open deals       : 8
Pipeline value   : Rs. 4180000
Forecast         : Rs. 1580000
Won revenue      : Rs. 1250000 ( 3 deals )
Lost deals       : 2
Win rate         : 60.0 %

--- Target: Rs. 3000000 ---
[########------------] 41.7 %

--- Deals by stage ---
Pipeline     4 deals | Rs. 1700000 | #################
Qualified    4 deals | Rs. 2480000 | ########################
Closed Won   3 deals | Rs. 1250000 | ############
Closed Lost  2 deals | Rs. 670000 | ######

--- Sales team ---
Sara    